# Flagging at-risk students in an unseen module

Loads the model persisted by `aga_fastpath_journeys.ipynb` step 15 and applies it to **a module
that was never part of training**, producing a ranked worklist of students likely to fail or
withdraw.

The recommended configuration is `journeyEmbedding + logClicks + submission` at **day 90** — top
precision in both modules it was measured on, 0.855 on BBB and 0.832 on GGG. See
`docs/model-selection.md` and `docs/assessment-submission.md`.

## What this notebook is actually testing

Applying a model across modules is an extrapolation, and this repository's own evidence says
absolute numbers do not transfer: the same code and hyperparameters scored 0.722 on GGG and 0.838
on BBB. So this notebook does **two** things:

1. produces the worklist, which is the deliverable, and
2. **scores itself against the real outcomes**, because OULAD has labels for every module.

The second is not optional. A ranked list of names is easy to produce and impossible to trust; the
only reason to believe it is to check what it would have got right. If transfer fails, the
measurement will say so rather than the list quietly being wrong.

A refit-on-target reference is also trained, so the cost of transferring rather than training
locally is visible instead of assumed.

## The trap this notebook is built around

`activityTypeId` is a **categorical** input to FastPath, and activity-type vocabularies differ by
module — GGG has 7, BBB 12, EEE 11, out of 20 in the dataset. If the vocabulary is enumerated per
module then `0` means `forumng` in GGG and `dualpane` in EEE, the two embedding spaces are
unrelated, and a transferred model produces confident nonsense with no error anywhere.

Step 4 therefore uses the **same global vocabulary** the model was trained with, and step 5 aborts
if it does not match.

## 1. Setup

In [ ]:
import os
import subprocess
import sys

REPO_URL = 'https://github.com/jose-alvarado-guzman/oulad.git'
REPO_DIR = '/content/oulad'

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

def run(*command):
    result = subprocess.run(command, text=True, capture_output=True)
    print((result.stdout + result.stderr).strip())
    result.check_returncode()

if IN_COLAB:
    if os.path.isdir(os.path.join(REPO_DIR, '.git')):
        run('git', '-C', REPO_DIR, 'fetch', '--depth', '1', 'origin', 'main')
        run('git', '-C', REPO_DIR, 'reset', '--hard', 'origin/main')
        run('git', '-C', REPO_DIR, 'clean', '-fd')
    else:
        run('git', 'clone', '--depth', '1', REPO_URL, REPO_DIR)
    run('git', '-C', REPO_DIR, 'log', '-1', '--format=%h %ad %s', '--date=short')
    print()
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', '-r',
         os.path.join(REPO_DIR, 'requirements-aga.txt')], check=True)
    print('Dependencies installed.')
else:
    REPO_DIR = os.getcwd()
    while REPO_DIR != '/' and not os.path.isdir(os.path.join(REPO_DIR, '.git')):
        REPO_DIR = os.path.dirname(REPO_DIR)
    print('Local kernel; assuming requirements-aga.txt is installed.')
    print('Repository root:', REPO_DIR)

## 2. Imports

In [ ]:
import os
import sys
from datetime import timedelta

REPO_DIR = '/content/oulad' if os.path.isdir('/content/oulad') else REPO_DIR
SRC_DIR = os.path.join(REPO_DIR, 'src')
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

for name in [m for m in sys.modules if m == 'oulad' or m.startswith('oulad.')]:
    del sys.modules[name]

import matplotlib.pyplot as plt
import pandas as pd
from neo4j import GraphDatabase
from graphdatascience.session import (
    AlgorithmCategory, AuraAPICredentials, DbmsConnectionInfo, GdsSessions,
    SessionMemory)

import graphdatascience
from oulad.credentials import (
    AGA_SECRETS, ETL_SECRETS, MissingCredentialsError, aura_instance_id, load_credentials)
from oulad.logger import get_logger

print('graphdatascience', graphdatascience.__version__)
print('repository      ', REPO_DIR)

## 3. Credentials and the database connection

In [ ]:
logger = get_logger(REPO_DIR)

try:
    print('resolved from:', load_credentials(logger, required=ETL_SECRETS + AGA_SECRETS))
except MissingCredentialsError as error:
    raise SystemExit(f'\n{error}\n\nAdd the missing secrets in the sidebar, switch on '
                     'Notebook access, then re-run this cell.')

NEO4J_URI = os.environ['NEO4J_URI']
NEO4J_USERNAME = os.environ['NEO4J_USERNAME']
NEO4J_PASSWORD = os.environ['NEO4J_PASSWORD']
NEO4J_DATABASE = os.getenv('NEO4J_DATABASE') or None
AURA_INSTANCE_ID = aura_instance_id(logger)

# liveness_check_timeout is not optional for this notebook. FastPath and training
# can leave the driver idle for tens of minutes, and a pooled connection that has
# gone stale surfaces as "Unable to retrieve routing information" or "Failed to
# read from defunct connection" in whichever cell happens to run next -- which on
# one run was the cleanup step, leaving 597,336 orphan Interaction nodes behind.
# With it set, an idle connection is verified before reuse.
driver = GraphDatabase.driver(
    NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD),
    liveness_check_timeout=30, max_connection_lifetime=600)
driver.verify_connectivity()
print('connected to AuraDB, instance', AURA_INSTANCE_ID)

sessions = GdsSessions(api_credentials=AuraAPICredentials(
    os.environ['AURA_CLIENT_ID'], os.environ['AURA_CLIENT_SECRET'],
    os.environ['AURA_PROJECT_ID']))

## 4. Choose the unseen module and load the stored model

`TARGET_MODULE` must be a module the stored model never saw. The persisted model was trained on
GGG; BBB was the second validation module, so both are excluded here.

**EEE** is the default: 2,634 students and 961k events, so the chain is a quarter the size of
BBB's, and its first assessment falls on day 33, which leaves six gradeable assessments by day 90 —
enough submission evidence for the features to mean something without being so dense that the
sequence embedding has nothing left to add.

In [ ]:
TARGET_MODULE = 'EEE'          # never used in training
TRAINED_ON = {'GGG', 'BBB'}    # what the stored model has seen
MODEL_NAME = 'oulad-atrisk-d90'
CUTOFF_DAY = 90                # must match the stored model
CONTACT_BUDGET = 100           # how many students the team can actually reach

FEATURES = ['journeyEmbedding', 'logClicks',
            'submissionRate', 'missedAll', 'missedFirst', 'meanLateness']
PASS_RESULTS = ['Pass', 'Distinction']

if TARGET_MODULE in TRAINED_ON:
    raise SystemExit(f'{TARGET_MODULE} was part of training. The point of this notebook is to '
                     'score a module the model has never seen; pick another.')

SPAN_QUERY = '''
MATCH (s:Student)-[r:REVIEWED_MATERIAL]->(m:EducationalMaterial)<-[:HAS_MATERIAL]-(c:Course)
WHERE c.codeModule = $module
RETURN count(DISTINCT s) AS students, count(r) AS events,
       min(r.date) AS minDate, max(r.date) AS maxDate
'''
# The SAME global vocabulary the model was trained with -- not scoped to the module.
TYPES_QUERY = '''
MATCH (m:EducationalMaterial)
RETURN DISTINCT m.activityType AS activityType ORDER BY activityType
'''
with driver.session(database=NEO4J_DATABASE) as session:
    span = session.run(SPAN_QUERY, module=TARGET_MODULE).single()
    activity_types = [r['activityType'] for r in session.run(TYPES_QUERY)]

TYPE_IDS = {name: index for index, name in enumerate(activity_types)}
SHIFT = -min(0, span['minDate'])
OBSERVATION_TIME = float(CUTOFF_DAY + SHIFT + 1)
LOOKBACK_HORIZON = int(OBSERVATION_TIME) + 10
NUM_TIME_ANCHORS = 20

print(f"target module {TARGET_MODULE}: {span['students']:,} students, "
      f"{span['events']:,} events, days {span['minDate']} to {span['maxDate']}")
print(f'cutoff day {CUTOFF_DAY} (shifted {CUTOFF_DAY + SHIFT}), '
      f'observation_time {OBSERVATION_TIME:.0f}')
print(f'\nglobal activity-type vocabulary, {len(TYPE_IDS)} types:')
print(f'  {TYPE_IDS}')
present = {r['activityType'] for r in [{'activityType': t} for t in activity_types]}
print(f'\nthis is deliberately the whole-dataset vocabulary, not the '
      f'{TARGET_MODULE} subset -- see the header')

## 5. Build the event chain for the target module

Identical construction to the training notebook, with the same global vocabulary and the same
day-90 cutoff. Anything that differs here changes the meaning of the embedding.

In [ ]:
BATCH = 100

CANDIDATES_QUERY = '''
MATCH (s:Student)-[r:REVIEWED_MATERIAL]->(:EducationalMaterial)<-[:HAS_MATERIAL]-(c:Course)
WHERE c.codeModule = $module AND NOT (s)-[:FIRST_INTERACTION]->(:Interaction)
  AND r.date <= $cutoff
RETURN DISTINCT s.id AS studentId ORDER BY studentId
'''
BUILD_QUERY = '''
UNWIND $studentIds AS studentId
MATCH (s:Student {id: studentId})-[r:REVIEWED_MATERIAL]->(m:EducationalMaterial)
      <-[:HAS_MATERIAL]-(c:Course)
WHERE c.codeModule = $module AND r.date <= $cutoff
WITH s, m, r ORDER BY r.date, m.id
WITH s, collect({material: m, day: r.date + $shift, clicks: r.sumClick,
                 typeId: $typeIds[m.activityType]}) AS events
UNWIND range(0, size(events) - 1) AS i
WITH s, i, events[i] AS event
WITH s, i, event.material AS material, event.day AS day,
     event.clicks AS clicks, event.typeId AS typeId
CREATE (ev:Interaction {module: $module, studentId: s.id, seq: i, day: day,
                        clicks: clicks, activityTypeId: typeId,
                        features: [log(toFloat(clicks) + 1.0)]})
CREATE (ev)-[:OF_MATERIAL]->(material)
WITH s, ev ORDER BY ev.seq
WITH s, collect(ev) AS chain
WITH s, chain, chain[0] AS firstEvent
CREATE (s)-[:FIRST_INTERACTION]->(firstEvent)
WITH chain
UNWIND range(0, size(chain) - 2) AS j
WITH chain[j] AS previous, chain[j + 1] AS following
CREATE (previous)-[:NEXT_INTERACTION]->(following)
RETURN count(*) AS links
'''

with driver.session(database=NEO4J_DATABASE) as session:
    pending = [r['studentId'] for r in session.run(
        CANDIDATES_QUERY, module=TARGET_MODULE, cutoff=CUTOFF_DAY)]
print(f'{len(pending):,} students need a chain')

for start in range(0, len(pending), BATCH):
    with driver.session(database=NEO4J_DATABASE) as session:
        session.run(BUILD_QUERY, studentIds=pending[start:start + BATCH],
                    module=TARGET_MODULE, shift=SHIFT, typeIds=TYPE_IDS,
                    cutoff=CUTOFF_DAY).consume()
    done = min(start + BATCH, len(pending))
    if done % (BATCH * 10) == 0 or done == len(pending):
        print(f'  {done:,}/{len(pending):,}', flush=True)
print('chain built' if pending else 'chain already present')

## 6. Features for the target module

The same three writes the training notebook makes: the outcome label, the click aggregate, and the
four submission scalars. The label is written **only so the notebook can score itself** in step 10
— it is not an input to any feature.

In [ ]:
LABEL_QUERY = '''
MATCH (s:Student)-[:WAS_REGISTERED]->(:StudentRegistration)-[cc:CONTAINS_COURSE]->(c:Course)
WHERE c.codeModule = $module
WITH DISTINCT s, cc.finalResult AS finalResult
OPTIONAL MATCH (s)-[r:REVIEWED_MATERIAL]->(:EducationalMaterial)<-[:HAS_MATERIAL]-(c2:Course)
WHERE c2.codeModule = $module AND r.date <= $cutoff
WITH s, finalResult, coalesce(sum(r.sumClick), 0) AS clicks
SET s.passed = CASE WHEN finalResult IN $passResults THEN 1 ELSE 0 END,
    s.logClicks = log(toFloat(clicks) + 1.0)
RETURN count(*) AS labelled
'''
SUBMISSION_QUERY = '''
MATCH (c:Course {codeModule: $module})-[:HAS_ASSESSMENT]->(a:Assessment)
WHERE a.date IS NOT NULL AND NOT isNaN(a.date)
  AND a.assessmentType <> 'Exam' AND a.date <= $cutoff
WITH c, collect(a) AS due, min(a.date) AS firstDue
WITH c, due, size(due) AS nDue, [x IN due WHERE x.date = firstDue] AS firstAssessments
MATCH (s:Student)-[:WAS_REGISTERED]->(:StudentRegistration)-[:CONTAINS_COURSE]->(c)
OPTIONAL MATCH (s)-[w:WAS_ASSESSED_IN]->(a2:Assessment)
WHERE a2 IN due AND w.dateSubmitted <= $cutoff
  AND (w.isBanked IS NULL OR w.isBanked = 0)
WITH s, nDue, firstAssessments, count(DISTINCT a2) AS nSubmitted,
     avg(w.dateSubmitted - a2.date) AS lateness,
     count(DISTINCT CASE WHEN a2 IN firstAssessments THEN a2 END) AS gotFirst
SET s.submissionRate = CASE WHEN nDue = 0 THEN 1.0
                            ELSE toFloat(nSubmitted) / nDue END,
    s.missedAll   = CASE WHEN nDue > 0 AND nSubmitted = 0 THEN 1 ELSE 0 END,
    s.missedFirst = CASE WHEN gotFirst = 0 THEN 1 ELSE 0 END,
    s.meanLateness = coalesce(lateness, 0.0)
RETURN count(*) AS scored, max(nDue) AS assessmentsDue
'''
with driver.session(database=NEO4J_DATABASE) as session:
    labelled = session.run(LABEL_QUERY, module=TARGET_MODULE, cutoff=CUTOFF_DAY,
                           passResults=PASS_RESULTS).single()['labelled']
    got = session.run(SUBMISSION_QUERY, module=TARGET_MODULE,
                      cutoff=CUTOFF_DAY).single()

print(f'labelled {labelled:,} students')
print(f"submission features on {got['scored']:,} students "
      f"({got['assessmentsDue']} assessments due by day {CUTOFF_DAY})")
if not got['assessmentsDue']:
    raise SystemExit(f'No assessment is due in {TARGET_MODULE} by day {CUTOFF_DAY}, so four of '
                     'the six features the model expects are constant. The transfer would be '
                     'meaningless. Pick a later cutoff or a different module.')

## 7. Open a session and project the chain

In [ ]:
COUNT_QUERY = '''
MATCH (i:Interaction {module: $module})
WITH count(i) AS events
MATCH (s:Student)-[:FIRST_INTERACTION]->(:Interaction {module: $module})
WITH events, count(DISTINCT s) AS students
MATCH (:Interaction {module: $module})-[n:NEXT_INTERACTION]->()
RETURN events, students, events + count(n) AS relationships
'''
with driver.session(database=NEO4J_DATABASE) as session:
    sized = session.run(COUNT_QUERY, module=TARGET_MODULE).single()
print(f"{sized['students']:,} students + {sized['events']:,} interactions")

memory = sessions.estimate(
    node_count=sized['events'] + sized['students'],
    relationship_count=sized['relationships'],
    algorithm_categories=[AlgorithmCategory.NODE_EMBEDDING],
    node_label_count=2, node_property_count=8)
# Same override as the training notebook: sessions.estimate() does not account
# for FastPath and returned m_2GB for a 113k-node chain that then aborted with
# "The job ran out of memory". Treat it as a floor to raise.
node_count = sized['events'] + sized['students']
FLOOR = [(400_000, SessionMemory.m_8GB),
         (1_000_000, SessionMemory.m_16GB),
         (float('inf'), SessionMemory.m_32GB)]
required = next(m for limit, m in FLOOR if node_count < limit)
if memory != required:
    print(f'overriding {memory} -> {required} for {node_count:,} nodes')
    memory = required
print('session memory:', memory)

SESSION_NAME = f"oulad-score-{os.environ['AURA_CLIENT_ID'][:8]}"
gds = sessions.get_or_create(
    session_name=SESSION_NAME, memory=memory,
    db_connection=DbmsConnectionInfo(
        aura_instance_id=AURA_INSTANCE_ID, username=NEO4J_USERNAME,
        password=NEO4J_PASSWORD, database=NEO4J_DATABASE),
    ttl=timedelta(hours=2))
print('session ready:', SESSION_NAME)

GRAPH_NAME = f'oulad-score-{TARGET_MODULE}'
PROJECTION_QUERY = '''
MATCH (src)-[r:FIRST_INTERACTION|NEXT_INTERACTION]->(tgt:Interaction)
WHERE tgt.module = $module
RETURN gds.graph.project.remote(src, tgt, {
    sourceNodeLabels: labels(src),
    targetNodeLabels: labels(tgt),
    sourceNodeProperties: src { .id, .day, .clicks, .activityTypeId, .features,
                                .passed, .logClicks,
                                .submissionRate, .missedAll, .missedFirst, .meanLateness },
    targetNodeProperties: tgt { .day, .clicks, .activityTypeId, .features },
    relationshipType: type(r)
})
'''
gds.graph.project.cypher(graph_name=GRAPH_NAME, query=PROJECTION_QUERY,
                         query_parameters={'module': TARGET_MODULE}, overwrite=True)
G = gds.graph.get(GRAPH_NAME)
print(f'projected {G.node_count():,} nodes, {G.relationship_count():,} relationships')

missing = set(FEATURES[1:]) - set(G.node_properties().get('Student', []))
if missing:
    raise SystemExit(f'Student is missing {missing} in the projection. The model would read '
                     'defaults for them and score confidently on nothing.')
print('every non-embedding feature is present')

## 8. FastPath, with the training notebook's parameters

Every argument here must match what produced the stored model. A different `embedding_dimension`
fails loudly; a different `num_time_anchors` or `smoothing_rate` does not — it just yields
coordinates that mean something else.

In [ ]:
embedding = gds.fast_path.mutate(
    G,
    base_node_label='Student',
    event_node_label='Interaction',
    mutate_property='journeyEmbedding',
    embedding_dimension=128,
    lookback_horizon=LOOKBACK_HORIZON,
    num_time_anchors=NUM_TIME_ANCHORS,
    event_node_categorical_properties=['activityTypeId'],
    event_node_feature_vector_property='features',
    event_node_time_property='day',
    first_relationship_type='FIRST_INTERACTION',
    next_relationship_type='NEXT_INTERACTION',
    observation_time=OBSERVATION_TIME,
    smoothing_window=2,
    smoothing_rate=10.0 / LOOKBACK_HORIZON,
    random_seed=42,
)
print(embedding)

## 9. Load the stored model and score

`gds.model.load()` pulls the persisted model into this session. If it is absent, run step 15 of
`aga_fastpath_journeys.ipynb` first — a session model does not survive its session, only a stored
one does.

In [ ]:
available = [m.model_name for m in gds.model.list()]
print('in this session:', available)

if MODEL_NAME not in available:
    try:
        print(gds.model.load(MODEL_NAME))
    except Exception as error:
        raise SystemExit(
            f'Could not load {MODEL_NAME!r}: {str(error)[:200]}\n\n'
            'Run step 15 of aga_fastpath_journeys.ipynb with CUTOFF_DAY = 90 to train and '
            'store it. A session model is discarded when its session ends; only '
            'gds.model.store() survives.')

model = gds.model.get(MODEL_NAME)
print('\nloaded:', MODEL_NAME)

predictions = model.predict_stream(G, target_node_labels=['Student'])
pcol = [c for c in predictions.columns
        if c != 'nodeId' and 'probab' not in c.lower()][-1]
prob_cols = [c for c in predictions.columns if 'probab' in c.lower()]

ids = gds.graph.node_properties.stream(G, 'id', node_labels=['Student'])
ids = ids.rename(columns={ids.columns[-1]: 'studentId'})[['nodeId', 'studentId']]
truth = gds.graph.node_properties.stream(G, 'passed', node_labels=['Student'])
truth = truth.rename(columns={truth.columns[-1]: 'passed'})[['nodeId', 'passed']]

scored = (predictions.rename(columns={pcol: 'predicted'})
          .merge(ids, on='nodeId').merge(truth, on='nodeId'))
if prob_cols:
    probs = predictions[prob_cols[0]]
    # class 0 is at risk; predict_proba columns are ordered by class
    scored['risk'] = [p[0] if hasattr(p, '__getitem__') else float(p) for p in probs]
else:
    scored['risk'] = 1.0 - scored.predicted
print(f'scored {len(scored):,} students in {TARGET_MODULE}')
scored.head()

## 10. The worklist, and whether it can be trusted

The deliverable is the top `CONTACT_BUDGET` students by risk. A ranked budget is used rather than
the classifier's own argmax threshold because intervention capacity is a fixed number of contacts,
and because argmax on an imbalanced target can predict the majority class for everyone and report
zero flagged — a fitting failure that reads as a result.

Because OULAD carries outcomes for every module, precision at that budget is measurable, and it is
the only reason to believe the list.

In [ ]:
ranked = scored.sort_values('risk', ascending=False).reset_index(drop=True)
worklist = ranked.head(CONTACT_BUDGET)

at_risk = int((scored.passed == 0).sum())
base_rate = at_risk / len(scored)


def at_k(frame, k):
    top = ranked.head(k)
    hit = int((top.passed == 0).sum())
    return {'k': k, 'precision': hit / k, 'recall': hit / at_risk if at_risk else 0.0,
            'lift': (hit / k) / base_rate if base_rate else 0.0}

print(f'{TARGET_MODULE}: {len(scored):,} students, {at_risk:,} at risk '
      f'(base rate {base_rate:.3f})')
print(f'\nTRANSFERRED MODEL, trained on GGG, never saw {TARGET_MODULE}:')
budgets = pd.DataFrame([at_k(ranked, k) for k in
                        [50, CONTACT_BUDGET, 200, 400] if k <= len(ranked)])
print(budgets.round(4).to_string(index=False))

argmax_flagged = int((scored.predicted == 0).sum())
argmax_hit = int(((scored.predicted == 0) & (scored.passed == 0)).sum())
print(f'\nat the model\'s own threshold: flags {argmax_flagged:,}, '
      f'precision {argmax_hit / argmax_flagged if argmax_flagged else 0:.3f}, '
      f'recall {argmax_hit / at_risk if at_risk else 0:.3f}')
if argmax_flagged == 0:
    print('  the transferred model flags nobody at argmax -- treat the ranking as the '
          'only usable output, and read step 11 before trusting it')

print(f'\n--- worklist: {CONTACT_BUDGET} highest-risk students in {TARGET_MODULE} ---')
print(worklist[['studentId', 'risk', 'predicted', 'passed']].head(20).to_string(index=False))

## 11. What did transferring cost?

A reference model trained on the target module itself, same features and split. The gap between
the two is the price of transfer, and it is the number that decides whether a stored model can be
pointed at a new module or whether each module needs its own.

This is the honest control. Without it, a mediocre transferred result looks like the ceiling.

In [ ]:
REFIT_NAME = f'oulad-refit-{TARGET_MODULE}'
REFIT_PIPE = f'oulad-refit-pipeline-{TARGET_MODULE}'
for drop in (lambda: gds.model.get(REFIT_NAME).drop(),
             lambda: gds.pipeline.node_classification.get(REFIT_PIPE).drop()):
    try:
        drop()
    except Exception:
        pass

pipe, _ = gds.pipeline.node_classification.create(REFIT_PIPE)
pipe.select_features(FEATURES)
pipe.configure_split(test_fraction=0.3, validation_folds=4)
pipe.add_logistic_regression(penalty=(0.001, 1.0), max_epochs=300)
pipe.add_random_forest(max_depth=(4, 16), number_of_decision_trees=200)
refit, _ = pipe.train(G, model_name=REFIT_NAME, metrics=['F1_MACRO', 'ACCURACY'],
                      target_property='passed', target_node_labels=['Student'],
                      random_seed=42)
print('refit metrics:', {m: v.get('test') for m, v in (refit.metrics() or {}).items()
                         if isinstance(v, dict)})

rp = refit.predict_stream(G, target_node_labels=['Student'])
rcol = [c for c in rp.columns if c != 'nodeId' and 'probab' not in c.lower()][-1]
rprob = [c for c in rp.columns if 'probab' in c.lower()]
rs = rp.rename(columns={rcol: 'predicted'}).merge(truth, on='nodeId')
rs['risk'] = ([p[0] for p in rp[rprob[0]]] if rprob else 1.0 - rs.predicted)
rranked = rs.sort_values('risk', ascending=False).reset_index(drop=True)

rows = []
for k in [50, CONTACT_BUDGET, 200, 400]:
    if k > len(ranked):
        continue
    t_hit = int((ranked.head(k).passed == 0).sum())
    r_hit = int((rranked.head(k).passed == 0).sum())
    rows.append({'k': k, 'transferred': t_hit / k, 'refit_on_target': r_hit / k,
                 'cost_of_transfer': (r_hit - t_hit) / k})
comparison = pd.DataFrame(rows)
print(f'\nprecision at a fixed budget, {TARGET_MODULE}')
print(comparison.round(4).to_string(index=False))
print(f'\nbase rate {base_rate:.4f} -- any column at or below this is worthless')

## 12. Clean up

In [ ]:
def db_execute(query, **params):
    """Run a write, rebuilding the driver if its connection has gone stale."""
    global driver
    for attempt in (1, 2):
        try:
            with driver.session(database=NEO4J_DATABASE) as session:
                return session.run(query, **params).single()
        except Exception as error:
            if attempt == 2:
                raise
            print(f'  reconnecting after: {str(error)[:70]}')
            try:
                driver.close()
            except Exception:
                pass
            driver = GraphDatabase.driver(
                NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD),
                liveness_check_timeout=30, max_connection_lifetime=600)

DELETE_CHAIN = True

for what, drop in [('refit model', lambda: gds.model.get(REFIT_NAME).drop()),
                   ('refit pipeline',
                    lambda: gds.pipeline.node_classification.get(REFIT_PIPE).drop())]:
    try:
        drop()
    except Exception as error:
        print(f'{what}: {str(error)[:70]}')

try:
    G.drop(); print('projection dropped')
except Exception as error:
    print('projection:', error)
try:
    gds.delete(); print('session deleted')
except Exception as error:
    print('session:', error)

if DELETE_CHAIN:
    DELETE_QUERY = '''
    MATCH (i:Interaction {module: $module})
    WITH i LIMIT $batch
    DETACH DELETE i
    RETURN count(*) AS deleted
    '''
    removed = 0
    while True:
        n = db_execute(DELETE_QUERY, module=TARGET_MODULE, batch=10000)['deleted']
        removed += n
        if n == 0:
            break
        if removed % 100000 == 0:
            print(f'  deleted {removed:,}', flush=True)
    print(f'removed {removed:,} interaction nodes')
    db_execute('MATCH (s:Student) WHERE s.passed IS NOT NULL '
               'REMOVE s.passed, s.logClicks RETURN 0 AS done')
    db_execute('MATCH (s:Student) WHERE s.submissionRate IS NOT NULL '
               'REMOVE s.submissionRate, s.missedAll, s.missedFirst, '
               's.meanLateness RETURN 0 AS done')
    print('removed the Student properties this notebook wrote')

with driver.session(database=NEO4J_DATABASE) as session:
    totals = session.run('MATCH (n) WITH count(n) AS nodes MATCH ()-[r]->() '
                         'RETURN nodes, count(r) AS relationships').single()
print(f"graph totals: {totals['nodes']:,} nodes, "
      f"{totals['relationships']:,} relationships")
print('expected after a clean run: 66,920 nodes, 8,818,076 relationships')

print(f'\nThe stored model {MODEL_NAME!r} is untouched -- it lives beyond any session. '
      f'Remove it with gds.model.delete({MODEL_NAME!r}) from a live session if needed.')

## How to read the result

**Compare `transferred` against `refit_on_target` in step 11, not against the numbers in
`docs/model-selection.md`.** Those were measured on GGG and BBB with a different holdout; the only
fair reference for a transferred model is one trained on the same module.

**Compare both against the base rate.** A worklist whose precision equals the base rate has sorted
nobody usefully, however respectable the number looks — on a module where 40% fail, 0.40 precision
is a random sample.

**If transfer holds**, a single stored model can be pointed at any module, and the day-90 scoring
run becomes a chain build plus a `model.load()`.

**If transfer fails**, that is a finding rather than a failure, and the likely reasons are worth
checking in order: the activity-type vocabulary (step 4 guards it), the FastPath parameters (step 8
must match exactly), and assessment density — a module whose first assessment falls late has thin
submission features by day 90, and four of the six inputs go nearly constant.

Either way the answer belongs in `docs/model-selection.md`, because cross-module transfer has never
been measured in this repository and both outcomes are worth recording.